In [ ]:
!pip uninstall numpy -y

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [ ]:
!pip install numpy==1.26.4

In [ ]:
import numpy as np
print('numpy:', np.__version__)
# Must show 1.26.4

import tensorflow as tf
print('tensorflow:', tf.__version__)
print('All working!')

numpy: 1.26.4
tensorflow: 2.21.0
All working!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE      = '/content/drive/MyDrive/FraudDetection'
DATA_PATH = os.path.join(BASE, 'data')
PROC_PATH = os.path.join(BASE, 'data', 'processed')
MOD_PATH  = os.path.join(BASE, 'models')

print('Ready!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready!


In [ ]:
import numpy as np
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt

# Page-Hinkley detector written from scratch
# No River library needed — avoids numpy conflict
class PageHinkley:
    def __init__(self, min_instances=50, delta=0.005,
                 threshold=50, alpha=0.9999):
        self.min_instances = min_instances
        self.delta         = delta
        self.threshold     = threshold
        self.alpha         = alpha
        self.n             = 0
        self.sum           = 0.0
        self.x_mean        = 0.0
        self.min_sum       = float('inf')
        self.drift_detected = False

    def update(self, x):
        self.n      += 1
        self.x_mean  = self.alpha * self.x_mean + (1 - self.alpha) * x
        self.sum    += x - self.x_mean - self.delta
        self.min_sum = min(self.min_sum, self.sum)

        if self.n >= self.min_instances:
            if (self.sum - self.min_sum) > self.threshold:
                self.drift_detected = True
            else:
                self.drift_detected = False
        else:
            self.drift_detected = False

# Load test data
X_test = np.load(os.path.join(PROC_PATH, 'X_test.npy'))
y_test = np.load(os.path.join(PROC_PATH, 'y_test.npy'))

# Load Neural Network
nn = tf.keras.models.load_model(
    os.path.join(MOD_PATH, 'neural_network.h5'))

# Load threshold
with open(os.path.join(MOD_PATH, 'best_threshold.pkl'), 'rb') as f:
    threshold = pickle.load(f)

print(f'Test samples: {len(X_test):,}')
print(f'Fraud cases:  {int(y_test.sum())}')
print(f'Fraud rate:   {y_test.mean()*100:.4f}%')
print(f'Threshold:    {threshold}')
print()
print('Page-Hinkley detector ready (built from scratch)')
print('No River library needed!')

Test samples: 56,962
Fraud cases:  98
Fraud rate:   0.1720%
Threshold:    0.25000000000000006

Page-Hinkley detector ready (built from scratch)
No River library needed!


In [ ]:
detector = PageHinkley(
    min_instances=50,
    delta=0.005,
    threshold=50,
    alpha=0.9999
)

print('Getting Neural Network predictions...')
y_probs = nn.predict(X_test).flatten()
y_preds = (y_probs >= threshold).astype(int)

print(f'Predictions done: {len(y_preds):,}')
print()

errors       = []
drift_points = []
error_rates  = []
window       = 200

print('Processing transactions one by one...')

for i, (pred, actual) in enumerate(zip(y_preds, y_test)):
    error = int(pred != actual)
    errors.append(error)
    detector.update(error)

    if detector.drift_detected:
        drift_points.append(i)
        print(f'  DRIFT ALERT at transaction #{i:,}')

    rate = np.mean(errors[max(0, i-window):i+1])
    error_rates.append(rate)

print()
print('='*55)
print('DRIFT DETECTION RESULTS')
print('='*55)
print(f'Total transactions:  {len(X_test):,}')
print(f'Total drift alerts:  {len(drift_points)}')
print(f'Overall error rate:  {np.mean(errors)*100:.4f}%')
if len(drift_points) > 0:
    print(f'Drift detected at transactions:')
    for dp in drift_points:
        print(f'  #{dp:,}')
else:
    print('No drift detected — model stable throughout')
print('='*55)

Getting Neural Network predictions...
1781/1781 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Predictions done: 56,962

Processing transactions one by one...

DRIFT DETECTION RESULTS
Total transactions:  56,962
Total drift alerts:  0
Overall error rate:  0.0597%
No drift detected — model stable throughout


In [ ]:
print('='*65)
print('AXD-FRAUD FRAMEWORK — COMPLETE SYSTEM SUMMARY')
print('='*65)
print()
print('DATASET:         Kaggle ULB Credit Card Fraud')
print('                 284,807 transactions, 492 fraud (0.17%)')
print()
print('PREPROCESSING:   StandardScaler + Stratified Split + SMOTE')
print('                 Train: 454,902 (50/50 balanced)')
print('                 Test:  56,961 (0.17% natural)')
print()
print('MODELS COMPARED:')
print('  Logistic Reg    F1: 0.1094  Precision: 0.0581  REJECTED')
print('  Random Forest   F1: 0.7511  Precision: 0.6748  GOOD')
print('  XGBoost         F1: 0.5317  Precision: 0.3777  REJECTED')
print('  Neural Network  F1: 0.8241  Precision: 0.8119  SELECTED')
print()
print('SELECTED MODEL:  Neural Network')
print('  Threshold:      0.35 (optimised)')
print('  Accuracy:       0.9994')
print('  Precision:      0.8119')
print('  Recall:         0.8367')
print('  F1-Score:       0.8241')
print('  ROC-AUC:        0.9686')
print('  PR-AUC:         0.8499')
print()
print('EXPLAINABILITY:  SHAP DeepExplainer')
print('                 Every prediction explained')
print('                 GDPR Article 22 compliant')
print()
print(f'DRIFT DETECTION: Page-Hinkley')
print(f'                 {len(drift_points)} alerts detected')
print(f'                 Error rate: {np.mean(errors)*100:.4f}%')
print()
print('DEPLOYMENT:      Flask REST API + React Dashboard')
print()
print('='*65)
print('ALL 6 NOTEBOOKS COMPLETE')
print('Next: Start Flask API then React Dashboard')
print('='*65)

AXD-FRAUD FRAMEWORK — COMPLETE SYSTEM SUMMARY

DATASET:         Kaggle ULB Credit Card Fraud
                 284,807 transactions, 492 fraud (0.17%)

PREPROCESSING:   StandardScaler + Stratified Split + SMOTE
                 Train: 454,902 (50/50 balanced)
                 Test:  56,961 (0.17% natural)

MODELS COMPARED:
  Logistic Reg    F1: 0.1094  Precision: 0.0581  REJECTED
  Random Forest   F1: 0.7511  Precision: 0.6748  GOOD
  XGBoost         F1: 0.5317  Precision: 0.3777  REJECTED
  Neural Network  F1: 0.8241  Precision: 0.8119  SELECTED

SELECTED MODEL:  Neural Network
  Threshold:      0.35 (optimised)
  Accuracy:       0.9994
  Precision:      0.8119
  Recall:         0.8367
  F1-Score:       0.8241
  ROC-AUC:        0.9686
  PR-AUC:         0.8499

EXPLAINABILITY:  SHAP DeepExplainer
                 Every prediction explained
                 GDPR Article 22 compliant

DRIFT DETECTION: Page-Hinkley
                 0 alerts detected
                 Error rate: 0.0597%

DE